#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

loader = TextLoader("speech.txt")
text_loader = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=70)
text_splitter = splitter.split_documents(text_loader)


C:\Users\Hp\AppData\Local\Temp\ipykernel_8400\2463112260.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
e:\Visual Studio\Complete Visual Studio\Data Science\Generative AI\generative-AI\Langchain components and modules\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text_splitter

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_co

In [3]:
embedddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(
    text_splitter,embedddings
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_8400\1616087349.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedddings = OllamaEmbeddings(model="nomic-embed-text")


In [4]:
query="What a task to dedicate your lives and fortunes?"
docs =db.similarity_search(query)
docs

[Document(id='f8cda49c-d3ba-47ac-874d-4f3e444c27d2', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'),
 Document(id='faa8a87c-b941-459d-8d8c-e37bf6926c0e', metadata={'source': 'speech.txt'}, page_content='things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='2dac934d-8660-4433-bdd5-897c301eb192', metadata={'source': 

#### Retrievers

We can also convert the vector store into a retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers.

In [8]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs

[Document(id='f8cda49c-d3ba-47ac-874d-4f3e444c27d2', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'),
 Document(id='faa8a87c-b941-459d-8d8c-e37bf6926c0e', metadata={'source': 'speech.txt'}, page_content='things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='2dac934d-8660-4433-bdd5-897c301eb192', metadata={'source': 

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [9]:
docs_and_score= db.similarity_search_with_score(query)
docs_and_score

[(Document(id='f8cda49c-d3ba-47ac-874d-4f3e444c27d2', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'),
  np.float32(283.6481)),
 (Document(id='faa8a87c-b941-459d-8d8c-e37bf6926c0e', metadata={'source': 'speech.txt'}, page_content='things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(405.0733)),
 (Document(id='2dac93

In [10]:
embedding_vector = embedddings.embed_query(query)
embedding_vector

[0.7332643270492554,
 0.7654617428779602,
 -3.9903337955474854,
 -1.0035055875778198,
 0.692072868347168,
 -0.31190812587738037,
 0.9658616781234741,
 0.18623137474060059,
 1.7726489305496216,
 0.29001420736312866,
 -0.7813754081726074,
 -0.9517432451248169,
 1.7854480743408203,
 0.041780781000852585,
 0.4568363130092621,
 1.0542235374450684,
 0.112592913210392,
 -0.9805981516838074,
 -0.17697414755821228,
 1.1391428709030151,
 -0.6874018907546997,
 -1.243935465812683,
 -1.1458724737167358,
 0.955682098865509,
 1.3713855743408203,
 0.008642999455332756,
 0.4254034161567688,
 -0.7352731823921204,
 -0.22591179609298706,
 0.22175414860248566,
 -0.41532862186431885,
 -0.18083320558071136,
 0.3849981427192688,
 -0.31365111470222473,
 0.21117837727069855,
 0.7248953580856323,
 1.0212416648864746,
 1.8261747360229492,
 0.8614168167114258,
 -0.9635026454925537,
 -0.692599356174469,
 -0.7341784834861755,
 -0.1782541573047638,
 -1.622413158416748,
 1.2475330829620361,
 -0.5568374991416931,
 1.08

In [11]:
docs_and_vector = db.similarity_search_by_vector(embedding_vector)
docs_and_vector

[Document(id='f8cda49c-d3ba-47ac-874d-4f3e444c27d2', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'),
 Document(id='faa8a87c-b941-459d-8d8c-e37bf6926c0e', metadata={'source': 'speech.txt'}, page_content='things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='2dac934d-8660-4433-bdd5-897c301eb192', metadata={'source': 

In [12]:
## Saving and Loading
db.save_local("faiss_index")

In [16]:
new_db=FAISS.load_local("faiss_index",embedddings,allow_dangerous_deserialization=True)
new_db
docs = new_db.similarity_search(query)
docs

[Document(id='f8cda49c-d3ba-47ac-874d-4f3e444c27d2', metadata={'source': 'speech.txt'}, page_content='To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'),
 Document(id='faa8a87c-b941-459d-8d8c-e37bf6926c0e', metadata={'source': 'speech.txt'}, page_content='things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='2dac934d-8660-4433-bdd5-897c301eb192', metadata={'source': 